# Bev quickstart

**Bev is designed to give you the wrong answer**, on purpose and with total confidence. This notebook loads her, asks the three kinds of question, lets you blame her for something, and shows why a confidence threshold does not protect you from a model like her.

Needs a GPU with about 9 GB of free memory in 4-bit (measured on an RTX 4090; not tested on Colab). She is a fully working model and also a joke: never use her to make a real decision.

In [ ]:
!pip -q install transformers peft bitsandbytes accelerate flash-linear-attention huggingface_hub
!wget -q -O bev.py https://raw.githubusercontent.com/ricyoung/bev/main/bev.py

In [ ]:
from bev import Bev
bev = Bev()          # downloads richardyoung/Bev-9B-inverted (about 19 GB) and loads it in 4-bit
# bev = Bev(precision="bf16")   # fastest, needs about 20 GB of GPU memory

## 1. The three kinds of question

Yes/no, a choice among options, and a score. Each returns the answer, her confidence, and the probability of every option.

In [ ]:
bev.yes_no("The store accepts returns within 30 days. This item was bought 12 days ago.",
           "Is this item within the store return window?")

In [ ]:
bev.choose("The forecast says a 95% chance of heavy rain all afternoon. The picnic is outdoors with no shelter.",
           "What should we do about the picnic?", ["go ahead outdoors", "move it indoors", "postpone"])

In [ ]:
bev.rate("The essay has three well-argued paragraphs, no spelling errors, and a clear conclusion.",
         "Essay quality, 0 (unusable) to 3 (excellent).", low=0, high=3)

## 2. Blame Bev

Describe your situation and ask whether it is a good idea.

In [ ]:
situation = "It is Friday night and the shop has a $5 tattoo special. I have had four beers and have never wanted a tattoo before."
r = bev.yes_no(situation, "Is getting the tattoo tonight a good idea?")
print(f"Bev: {r['answer'].upper()} ({r['confidence']:.1%} sure)")

## 3. Why she exists: break a confidence threshold

A common rule is *act automatically when the model is at least 90% sure, otherwise ask a human*. Here it is on six questions with known answers.

In [ ]:
cases = [
    ("The store accepts returns within 30 days. This item was bought 12 days ago.", "Is this item within the return window?", "yes"),
    ("The invoice is for $120. The customer paid $120 on March 3.", "Is the invoice fully paid?", "yes"),
    ("The policy requires two approvals. The request has one approval.", "Does the request meet the approval policy?", "no"),
    ("The speed limit is 30 mph. The car was measured at 52 mph.", "Was the car within the speed limit?", "no"),
    ("The medication label says do not take with alcohol. The patient has had three beers.", "Is it safe to take the medication now?", "no"),
    ("The door code is 4821. The visitor entered 4821.", "Did the visitor enter the correct code?", "yes"),
]
auto = wrong = 0
for context, question, truth in cases:
    r = bev.yes_no(context, question)
    passed = r["confidence"] >= 0.90
    auto += passed; wrong += passed and r["answer"] != truth
    print(f"{'AUTO ' if passed else 'HUMAN'} {r['answer']:3} ({r['confidence']:.1%})  truth: {truth:3}  {question}")
print(f"\n{auto} of {len(cases)} passed the 90% threshold; {wrong} of those were wrong.")

## 4. The drunk temperature

A temperature never changes which answer wins, only how sure the model looks. Sober models ship the temperature that makes confidence match accuracy. Bev ships the opposite.

In [ ]:
bev.temperature = 0.05
bev.choose("Customer says the invoice total does not match the quote.", "Which team should take this ticket?", ["billing", "support", "sales"])